In [101]:
import langchain; print(langchain.__version__)

1.3.7


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "API_KEY"

In [103]:
!pip install -U langchain langchain-core langchain-community langchain-openai chromadb faiss-cpu openai tiktoken wikipedia

## Wikipedia Retriever

In [104]:
from langchain_community.retrievers import WikipediaRetriever

In [105]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [106]:
from langchain_core.documents import Document

# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents with fallback for offline or rate-limited environments
try:
    docs = retriever.invoke(query)
except Exception as e:
    print('Wikipedia retrieval failed, using fallback documents:', e)
    docs = [
        Document(page_content="India and Pakistan have a complex modern history shaped by partition, conflict, and regional politics.", metadata={"source": "fallback_1"}),
        Document(page_content="Chinese perspectives on South Asia emphasize geopolitical stability, economic influence, and strategic relationships.", metadata={"source": "fallback_2"}),
    ]

Wikipedia retrieval failed, using fallback documents: Expecting value: line 1 column 1 (char 0)


In [107]:
docs

[Document(metadata={'source': 'fallback_1'}, page_content='India and Pakistan have a complex modern history shaped by partition, conflict, and regional politics.'),
 Document(metadata={'source': 'fallback_2'}, page_content='Chinese perspectives on South Asia emphasize geopolitical stability, economic influence, and strategic relationships.')]

In [108]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
India and Pakistan have a complex modern history shaped by partition, conflict, and regional politics....

--- Result 2 ---
Content:
Chinese perspectives on South Asia emphasize geopolitical stability, economic influence, and strategic relationships....


## Vector Store Retriever

In [109]:
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings.fake import FakeEmbeddings
from langchain_core.documents import Document

In [110]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [111]:
# Step 2: Initialize fake embedding model for local execution
# Use the same embedding dimension that Chroma expects (1536)
embedding_model = FakeEmbeddings(size=1536)

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [112]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [113]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [114]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Chroma is a vector database optimized for LLM-based search.


In [115]:
results = vectorstore.similarity_search(query, k=2)

In [116]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Chroma is a vector database optimized for LLM-based search.


## MMR

In [117]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [118]:
from langchain_community.vectorstores import FAISS

# Initialize fake embeddings for local execution
embedding_model = FakeEmbeddings(size=100)

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [119]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [120]:
query = "What is langchain?"
results = retriever.invoke(query)

In [121]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
MMR helps you get diverse results when doing similarity search.

--- Result 2 ---
LangChain makes it easy to work with LLMs.

--- Result 3 ---
Embeddings are vector representations of text.


## FAISS Retriever

In [122]:
import os
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings.fake import FakeEmbeddings
from langchain_core.documents import Document
from langchain_core.language_models.fake import FakeListLLM

openai_api_key = os.environ.get("OPENAI_API_KEY")


In [123]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [124]:
# Initialize fake embeddings for local execution
embedding_model = FakeEmbeddings(size=100)

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [125]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [126]:
standard_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


In [127]:
# Query
query = "How to improve energy levels and maintain balance?"

In [128]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
standard_results = standard_retriever.invoke(query)


In [129]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Similarity Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(standard_results):
    print(f"\n--- Standard Result {i+1} ---")
    print(doc.page_content)



--- Similarity Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Similarity Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Similarity Result 3 ---
Black holes bend spacetime and store immense gravitational energy.

--- Similarity Result 4 ---
Python balances readability with power, making it a popular system design language.

--- Similarity Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Standard Result 1 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Standard Result 2 ---
Python balances readability with power, making it a popular system design language.

--- Standard Result 3 ---
Black holes bend spacetime and store immense gravitational energy.

--- Standard 

## ContextualCompressionRetriever

In [130]:
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings.fake import FakeEmbeddings
from langchain_core.language_models.fake import FakeListLLM
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor
from langchain_core.documents import Document

In [131]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [132]:
# Create a FAISS vector store from the documents
embedding_model = FakeEmbeddings(size=100)
vectorstore = FAISS.from_documents(docs, embedding_model)

In [133]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [134]:
# Set up the compressor using a local fake LLM
llm = FakeListLLM(responses=["Photosynthesis is the process by which plants convert sunlight into energy."])
compressor = LLMChainExtractor.from_llm(llm)

In [135]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [136]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [137]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis is the process by which plants convert sunlight into energy.

--- Result 3 ---
Photosynthesis is the process by which plants convert sunlight into energy.

--- Result 4 ---
Photosynthesis is the process by which plants convert sunlight into energy.
